In [77]:
import tensorflow as tf
from keras import layers
import numpy as np
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input

In [78]:
gpus = tf.config.list_physical_devices('GPU')
print("GPUs detected:", gpus)

GPUs detected: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [79]:
train_ds = tf.keras.utils.image_dataset_from_directory(
    "dataset/training_set",         
    labels="inferred",        
    label_mode="int",         
    color_mode="rgb",         
    batch_size=32,
    image_size=(150,150),   
    seed=123,

    
)

Found 8000 files belonging to 2 classes.


In [80]:
validate_ds = tf.keras.utils.image_dataset_from_directory(
    "dataset/test_set",         
    labels="inferred",        
    label_mode="int",         
    color_mode="rgb",         
    batch_size=32,
    image_size=(150,150),   
    seed=123,

)

Found 2000 files belonging to 2 classes.


In [81]:
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomZoom(0.2),
    layers.RandomRotation(0.1),
    layers.RandomContrast(0.2)
])

In [108]:
model = Sequential([
    Input(shape=(150, 150, 3)),
    layers.Rescaling(1./255),
    data_augmentation,
    
    layers.Conv2D(32, (3,3), activation="relu"),
    layers.MaxPooling2D((2,2)),
    
    layers.Conv2D(64, (3,3), activation="relu"),
    layers.MaxPooling2D((2,2)),

    layers.Conv2D(128, (3,3), activation="relu"),
    layers.MaxPooling2D((2,2)),

    layers.Conv2D(256, (3,3), activation="relu"),
    layers.MaxPooling2D((2,2)),

    layers.Flatten(),
    layers.Dense(128, activation="relu"),
    layers.Dense(1, activation="sigmoid"),
])


In [109]:
model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=['accuracy']
)

In [110]:
model.fit(
    train_ds,
    validation_data=validate_ds,
    epochs=30
)


Epoch 1/30
250/250 ━━━━━━━━━━━━━━━━━━━━ 12s 40ms/step - accuracy: 0.5277 - loss: 0.6980 - val_accuracy: 0.5000 - val_loss: 0.6926
Epoch 2/30
250/250 ━━━━━━━━━━━━━━━━━━━━ 9s 37ms/step - accuracy: 0.5255 - loss: 0.6902 - val_accuracy: 0.6065 - val_loss: 0.6570
Epoch 3/30
250/250 ━━━━━━━━━━━━━━━━━━━━ 11s 42ms/step - accuracy: 0.5983 - loss: 0.6653 - val_accuracy: 0.6040 - val_loss: 0.6655
Epoch 4/30
250/250 ━━━━━━━━━━━━━━━━━━━━ 11s 42ms/step - accuracy: 0.6461 - loss: 0.6379 - val_accuracy: 0.6575 - val_loss: 0.6280
Epoch 5/30
250/250 ━━━━━━━━━━━━━━━━━━━━ 10s 40ms/step - accuracy: 0.6877 - loss: 0.5929 - val_accuracy: 0.7015 - val_loss: 0.5740
Epoch 6/30
250/250 ━━━━━━━━━━━━━━━━━━━━ 30s 119ms/step - accuracy: 0.7242 - loss: 0.5491 - val_accuracy: 0.7390 - val_loss: 0.5237
Epoch 7/30
250/250 ━━━━━━━━━━━━━━━━━━━━ 16s 63ms/step - accuracy: 0.7438 - loss: 0.5225 - val_accuracy: 0.7675 - val_loss: 0.4931
Epoch 8/30
250/250 ━━━━━━━━━━━━━━━━━━━━ 12s 46ms/step - accuracy: 0.7450 - loss: 0.5095 - 

accuracy: 0.9854 - loss: 0.0446 - val_accuracy: 0.7520 - val_loss: 1.4706

In [111]:
model.save("model.keras", overwrite=True)

In [112]:
from keras.preprocessing import image
from tensorflow.keras.models import load_model
loaded_model = load_model("model.keras")

In [119]:
test_image = image.load_img('dataset/new_images/dog.jpg',target_size=(150, 150))
test_image = image.img_to_array(test_image)
test_image = np.expand_dims(test_image,axis = 0)

result = loaded_model.predict(test_image)
print(result)
class_names = train_ds.class_names  
print(class_names)

if result[0][0] > 0.5:
    prediction = class_names[1]   
else:
    prediction = class_names[0]  
print("Predicted:", prediction)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step
[[0.9881706]]
['cats', 'dogs']
Predicted: dogs


In [ ]:
test_image = image.load_img('dataset/new_images/cat.jpg',target_size=(150, 150))
test_image = image.img_to_array(test_image)
test_image = np.expand_dims(test_image,axis = 0)

result = loaded_model.predict(test_image)
print(result)
class_names = train_ds.class_names  
print(class_names)

if result[0][0] > 0.5:
    prediction = class_names[1]   
else:
    prediction = class_names[0]  
print("Predicted:", prediction)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step
[[0.00135102]]
['cats', 'dogs']
Predicted: cats
